# 04 · Execution, runtime guardrails and step records

**Role of this layer.** The executor is the only code that calls capability
functions. It re-validates the plan (it does not trust its caller), refuses to
start while any BLOCK exists or any APPROVE issue is unapproved, then runs the
steps in order with the runtime guardrails that need data (enforced by the executor):

| code | action | checks |
|---|---|---|
| E01 | BLOCK | snapshot file hash still matches the catalog |
| E02 | BLOCK | enough observations for the capability (e.g. 3 x window) |
| E03 | WARN | more than 5% of a raw series missing |
| E04 | WARN | an inner join drops more than 10% of an input |
| E05 | WARN | the flagged group of a comparison or test has fewer than 5 distinct episodes |

Every step produces a **step record**: parameters used (defaults filled), input
hashes, output hash, sample size and range, duration, issues.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.data import load_catalog
from quantlab.executor import execute_plan
from quantlab.mock_scenarios import SCENARIOS_BY_ID
from quantlab.plan import Plan
from quantlab.validator import fill_defaults

catalog = load_catalog(settings.catalog_path)


def scenario(scenario_id):
    s = SCENARIOS_BY_ID[scenario_id]
    return fill_defaults(Plan(plan_id="demo", request=s.request, **s.plan))


plan_5 = scenario("buy_the_dip_tech")
refused = execute_plan(plan_5, catalog, approved_issue_ids=set())
print(refused.status, "-", refused.error)

blocked - Approval required for: V10:s3:return_method, V10:s5:return_method


With the approval granted, scenario 5 runs on the real NASDAQ snapshot:

In [3]:
approvals = {"V10:s3:return_method", "V10:s5:return_method"}  # returns and forward returns
result = execute_plan(plan_5, catalog, approved_issue_ids=approvals)
print("status:", result.status, f"({result.duration_ms:.0f} ms)")
pd.DataFrame(
    [
        {
            "step": r.step_id,
            "code": r.capability_code,
            "capability": r.capability,
            "params": r.params,
            "n_obs": r.n_obs,
            "range": f"{r.start}..{r.end}",
            "ms": r.duration_ms,
            "output sha256": r.output_hash[:12],
        }
        for r in result.step_records
    ]
)

status: completed (145 ms)


,step,code,capability,params,n_obs,range,ms,output sha256
0,s1,C01,load_series,{'dataset_id': 'NASDAQCOM'},14042,1971-02-05..2026-10-15,71.81,19533e0349f1
1,s2,C02,resample,"{'freq': 'W', 'how': 'last'}",2906,1971-02-05..2026-10-09,31.22,0c44f38c8ce1
2,s3,C03,compute_returns,"{'method': 'simple', 'periods': 1}",2905,1971-02-12..2026-10-09,12.65,b607a740b787
3,s4,C07,flag_regime,"{'quantile': 0.1, 'method': 'expanding', 'min_periods': 52, 'direction': 'below'}",2853,1972-02-11..2026-10-09,10.79,f7a9b4c68ffd
4,s5,C08,forward_return,{'method': 'simple'},2905,1971-02-05..2026-10-02,12.78,b4b3370e2ab0
5,s6,C10,hac_mean_difference,{},2852,1972-02-11..2026-10-02,6.03,c9e0713c6401


The fourth step holds the flags: True for the dips, the weeks below the 10th percentile
of past weekly returns.

In [4]:
flags = result.artifacts["s4"].value
print(f"{int(flags.sum())} of {len(flags)} weeks flagged")
flags[flags].tail()

334 of 2853 weeks flagged


date
2025-04-04    True
2025-11-07    True
2026-03-27    True
2026-06-05    True
2026-06-26    True
Name: NASDAQCOM, dtype: bool

The last step is the significance test: the next week's simple return after a dip
versus after the other weeks, with Newey-West standard errors (all its numbers are in one table):

In [5]:
pd.Series(result.artifacts["s6"].value, name="s6 hac_mean_difference")

diff_means             -0.003995
mean_flagged           -0.001243
mean_normal             0.002752
se_hac                  0.002186
t_stat                  -1.82795
p_value                 0.067557
ci_low                 -0.008279
ci_high                 0.000289
n_obs                       2852
n_flagged                    334
n_normal                    2518
n_episodes_flagged           258
maxlags_used                   8
horizon                        1
start                 1972-02-11
end                   2026-10-02
Name: s6 hac_mean_difference, dtype: object

**E02**: a 250-month window on monthly data does not have 3 x 250 observations.

In [6]:
from quantlab.plan import Step

plan = fill_defaults(
    Plan(
        plan_id="demo",
        request="monthly volatility",
        dataset_ids=["NASDAQCOM"],
        steps=[
            Step(id="s1", capability="load_series", params={"dataset_id": "NASDAQCOM"}),
            Step(id="s2", capability="resample", inputs={"series": "s1"}, params={"freq": "M"}),
            Step(
                id="s3",
                capability="compute_returns",
                inputs={"series": "s2"},
                params={"method": "simple"},
            ),
            Step(
                id="s4",
                capability="rolling_volatility",
                inputs={"series": "s3"},
                params={"window": 250},
            ),
        ],
    )
)
blocked = execute_plan(plan, catalog, {"V10:s3:return_method"})
print(blocked.status, "-", blocked.error)

blocked - Step s4 (rolling_volatility): rolling_volatility has 667 usable observations; it needs 750 (at least 3 x window observations).


**E04** fires on the real data for the 10y-2y spread: the 2-year yield starts in
1976 and the 10-year in 1962, so the spread silently loses 22% of the 10-year
history. The run completes, with the warning recorded.

In [7]:
spread = fill_defaults(
    Plan(
        plan_id="demo",
        request="10y-2y spread",
        steps=[
            Step(id="s1", capability="load_series", params={"dataset_id": "DGS10"}),
            Step(id="s2", capability="load_series", params={"dataset_id": "DGS2"}),
            Step(id="s3", capability="spread", inputs={"a": "s1", "b": "s2"}),
        ],
    )
)
result_e = execute_plan(spread, catalog)
print(result_e.status)
for issue in result_e.issues:
    print(f"[{issue.action}] {issue.code}: {issue.message}")

completed
[WARN] E04: Aligning the inputs kept 12590 of 16182 rows of 'a' (22% dropped).


Determinism: running the same plan twice gives identical output hashes.

In [8]:
again = execute_plan(spread, catalog)
[r.output_hash for r in again.step_records] == [r.output_hash for r in result_e.step_records]

True